In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GAE
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from torch_geometric.utils import from_scipy_sparse_matrix
from torch_geometric.data import Data
import scipy.sparse as sp
import networkx as nx
import matplotlib.pyplot as plt
from torch.nn import BatchNorm1d
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.nn import BatchNorm1d
import matplotlib.pyplot as plt
import numpy as np
from torch_geometric.utils import dropout_adj
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


In [3]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv, BatchNorm

class GATEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, heads=4, dropout_rate=0.01):
        super(GATEncoder, self).__init__()
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate
        
        # First layer: input to hidden size with multi-head attention
        self.convs = torch.nn.ModuleList()
        self.convs.append(GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout_rate))
        
        # Add intermediate layers with attention
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(heads * hidden_channels, hidden_channels, heads=heads, dropout=dropout_rate))
        
        # Final layer: hidden size to output size
        self.convs.append(GATConv(heads * hidden_channels, out_channels, heads=1, concat=False, dropout=dropout_rate))  # Single head for final layer
    
        # Batch normalization layers for each convolution layer
        self.bns = torch.nn.ModuleList([BatchNorm(hidden_channels * heads) for _ in range(num_layers - 1)])
    
        # Dropout rate
        self.dropout = torch.nn.Dropout(p=dropout_rate)
    
        # LeakyReLU instead of ReLU
        self.activation = torch.nn.LeakyReLU(negative_slope=0.05)
    
    def forward(self, x, edge_index):
        dead_neurons = []  # To track dead neurons across layers

        # First layer (without batch normalization)
        x = self.convs[0](x, edge_index)
        activations_1 = (x == 0).float().mean()  # Track how many activations are zero
        dead_neurons.append(activations_1.item())
        x = self.activation(x)
    
        # Intermediate layers with batch normalization and dropout
        for i in range(1, self.num_layers - 1):
            x_residual = x  # For residual connection
            x = self.convs[i](x, edge_index)
            x = self.bns[i - 1](x)  # Apply batch normalization
            activations_mid = (x == 0).float().mean()  # Track zero activations in hidden layers
            dead_neurons.append(activations_mid.item())
            #x = self.activation(x)
            x = F.relu(x)
            x = self.dropout(x)
            
            # Add residual connection (skip connection)
            x = x + x_residual
            
        # Final layer (no activation or batch norm)
        x = self.convs[-1](x, edge_index)
        activations_final = (x == 0).float().mean()  # Track zero activations in the final layer
        dead_neurons.append(activations_final.item())
        x = F.relu(x)

        # Print dead neurons for each layer
        #print(f"Dead neurons (percentage of activations that are zero) by layer: {dead_neurons}")
        return x



In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, BatchNorm

class GATEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, heads=4, dropout_rate=0.01, decode_type=None):
        super(GATEncoder, self).__init__()
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate
        self.heads = heads
        self.decode_type = decode_type  # Choose the decoding type: 'dot', 'cosine', 'MLP', or None (for no decoding)

        # GATConv layers (from GATEncoder)
        self.convs = nn.ModuleList()
        self.convs.append(GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout_rate))
        
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(heads * hidden_channels, hidden_channels, heads=heads, dropout=dropout_rate))
        
        # Final GATConv layer
        self.convs.append(GATConv(heads * hidden_channels, out_channels, heads=1, concat=False, dropout=dropout_rate))  # Single head for final layer

        # Batch Normalization layers (from GATEncoder)
        self.bns = nn.ModuleList([BatchNorm(hidden_channels * heads) for _ in range(num_layers - 1)])

        # A projection layer to align dimensions for residual connection
        self.proj = nn.Linear(in_channels, hidden_channels * heads)

        # MLP Decoder for interactions (from GENELink)
        if decode_type == 'MLP':
            self.linear = nn.Linear(2 * out_channels, 2)

        # Dropout rate
        self.dropout = nn.Dropout(p=dropout_rate)

        # LeakyReLU instead of ReLU (common in both)
        #self.activation = nn.LeakyReLU(negative_slope=0.05)
        x = F.relu(x)

    def forward(self, x, edge_index):
        # Project input x to hidden dimensions to ensure residual connection compatibility
        x_residual = self.proj(x)
        
        # Forward pass through GAT layers
        for i in range(self.num_layers - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)  # Apply batch normalization
            #x = self.activation(x)
            x = F.relu(x)
            x = self.dropout(x)
            x = x + x_residual  # Add residual connection
        
        # Final layer (no batch norm here)
        x = self.convs[-1](x, edge_index)
        x = F.relu(x)

        return x

    def decode(self, tf_embed, target_embed):
        # Decoding between two embeddings (borrowed from GENELink)
        if self.decode_type == 'dot':
            prob = torch.mul(tf_embed, target_embed)
            prob = torch.sum(prob, dim=1).view(-1, 1)
            return prob

        elif self.decode_type == 'cosine':
            prob = torch.cosine_similarity(tf_embed, target_embed, dim=1).view(-1, 1)
            return prob

        elif self.decode_type == 'MLP':
            h = torch.cat([tf_embed, target_embed], dim=1)
            prob = self.linear(h)
            return prob
        else:
            raise TypeError(f'{self.decode_type} is not available as a decoding option.')

    def forward_with_interactions(self, x, edge_index, train_sample=None):
        # Encode the gene expression (embedding) with GAT layers
        embed = self.forward(x, edge_index)

        if train_sample is not None and self.decode_type is not None:
            # Extract embeddings for TFs and targets
            tf_embed = embed[train_sample[:, 0]]
            target_embed = embed[train_sample[:, 1]]

            # Decode the interaction probability between TF and target embeddings
            pred = self.decode(tf_embed, target_embed)

            return pred
        else:
            # If no TF-target pairs are provided, return only embeddings
            return embed

    def get_embeddings(self, x, edge_index):
        # Function to return node embeddings after forward pass (to be used externally)
        embed = self.forward(x, edge_index)
        return embed



In [227]:
class GATEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, heads=4, dropout_rate=0.01, decode_type=None):
        super(GATEncoder, self).__init__()
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate
        self.heads = heads
        self.decode_type = decode_type  # Choose the decoding type: 'dot', 'cosine', 'MLP', or None (for no decoding)

        # GATConv layers (from GATEncoder)
        self.convs = nn.ModuleList()
        self.convs.append(GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout_rate))
        
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(heads * hidden_channels, hidden_channels, heads=heads, dropout=dropout_rate))
        
        # Final GATConv layer
        self.convs.append(GATConv(heads * hidden_channels, out_channels, heads=1, concat=False, dropout=dropout_rate))  # Single head for final layer

        # Batch Normalization layers (from GATEncoder)
        self.bns = nn.ModuleList([BatchNorm(hidden_channels * heads) for _ in range(num_layers - 1)])

        # A projection layer to align dimensions for residual connection
        self.proj = nn.Linear(in_channels, hidden_channels * heads)

        # MLP Decoder for interactions (from GENELink)
        if decode_type == 'MLP':
            self.linear = nn.Linear(2 * out_channels, 2)

        # Dropout rate
        self.dropout = nn.Dropout(p=dropout_rate)

        # LeakyReLU instead of ReLU (common in both)
        self.activation = nn.LeakyReLU(negative_slope=0.05)

    def forward(self, x, edge_index):
        # Project input x to hidden dimensions to ensure residual connection compatibility
        x_residual = self.proj(x)
        
        # Forward pass through GAT layers
        for i in range(self.num_layers - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)  # Apply batch normalization
            x = self.activation(x)
            x = self.dropout(x)
            x = x + x_residual  # Add residual connection
        
        # Final layer (no batch norm here)
        x = self.convs[-1](x, edge_index)
        x = F.relu(x)

        return x


In [19]:
def apply_gae_to_adjacency(adj_matrix, node_features, embedding_dim=4, epochs=200, dropout_rate=0.01, weight_decay=1e-4):
    # Convert adjacency matrix to sparse format
    adj_sparse = sp.csr_matrix(adj_matrix)
    adj_normalized = normalize_adj(adj_sparse + sp.eye(adj_sparse.shape[0]))
    edge_index, edge_weight = from_scipy_sparse_matrix(adj_normalized)
    
    # Create data object
    x = torch.tensor(node_features, dtype=torch.float)
    data = Data(x=x, edge_index=edge_index)

    # Define the GATEncoder as the encoder for the GAE model
    encoder = GATEncoder(in_channels=x.shape[1], hidden_channels=embedding_dim, out_channels=embedding_dim, dropout_rate=dropout_rate)

    # Instantiate the GAE model by passing the encoder
    model = GAE(encoder)

    # Training settings
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    data = data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=weight_decay)

    # Metrics to track
    losses = []
    accuracies = []

    # Training loop
    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()

        # Forward pass
        z = model.encode(data.x, data.edge_index)

        # Reconstruction loss
        loss = model.recon_loss(z, data.edge_index)

        # Accuracy: Compare reconstruction with ground truth adjacency matrix
        adj_pred = (model.decoder.forward_all(z) > 0.5).float()  # Example threshold for prediction
        acc = (adj_pred == torch.tensor(adj_matrix).float()).sum() / adj_pred.numel()  # Accuracy for all elements
        
        # Backward pass
        loss.backward()
        optimizer.step()

        # Store metrics
        losses.append(loss.item())
        accuracies.append(acc.item())

        # Print metrics every 50 epochs
        if epoch % 50 == 0:
            print(f'Epoch {epoch} | Loss: {loss.item():.4f} | Accuracy: {acc.item() * 100:.2f}%')

    # After training, reconstruct adjacency matrix
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index)
        adj_reconstructed = model.decoder.forward_all(z).cpu().numpy()
    
    return adj_reconstructed, losses, accuracies


In [7]:
def apply_gae_to_adjacency(adj_matrix, node_features, embedding_dim=4, epochs=200, dropout_rate=0.01, weight_decay=1e-4):
    # Convert adjacency matrix to sparse format
    adj_sparse = sp.csr_matrix(adj_matrix)
    adj_normalized = normalize_adj(adj_sparse + sp.eye(adj_sparse.shape[0]))
    edge_index, edge_weight = from_scipy_sparse_matrix(adj_normalized)
    
    # Create data object
    x = torch.tensor(node_features, dtype=torch.float)
    data = Data(x=x, edge_index=edge_index)

    # Ensure edge_index matches valid node indices
    max_index = torch.max(edge_index)
    print(f"edge_index is valid with max index: {max_index} and node count: {x.shape[0]}")
    assert max_index < x.shape[0], f"Edge index contains invalid nodes. Max index is {max_index}, but there are only {x.shape[0]} nodes."

    # Define the GATEncoder as the encoder for the GAE model
    encoder = GATEncoder(in_channels=x.shape[1], hidden_channels=embedding_dim, out_channels=embedding_dim, dropout_rate=dropout_rate)

    # Instantiate the GAE model by passing the encoder
    model = GAE(encoder)

    # Training settings
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    data = data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=weight_decay)

    # Metrics to track
    losses = []
    accuracies = []

    # Training loop
    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()

        # Forward pass
        z = model.encode(data.x, data.edge_index)

        # Reconstruction loss
        loss = model.recon_loss(z, data.edge_index)

        # Accuracy: Compare reconstruction with ground truth adjacency matrix
        adj_pred = (model.decoder.forward_all(z) > 0.5).float()  # Example threshold for prediction
        acc = (adj_pred == torch.tensor(adj_matrix).float()).sum() / adj_pred.numel()  # Accuracy for all elements

        # Backward pass
        loss.backward()
        optimizer.step()

        # Store metrics
        losses.append(loss.item())
        accuracies.append(acc.item())

        # Print metrics every 50 epochs
        if epoch % 50 == 0:
            print(f'Epoch {epoch} | Loss: {loss.item():.4f} | Accuracy: {acc.item() * 100:.2f}%')

    # After training, reconstruct adjacency matrix
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index)
        adj_reconstructed = model.decoder.forward_all(z).cpu().numpy()

    return adj_reconstructed, losses, accuracies


In [21]:

# Normalize the adjacency matrix
def normalize_adj(adj):
    """Symmetrically normalize adjacency matrix."""
    adj = sp.coo_matrix(adj)
    rowsum = np.array(adj.sum(1)).flatten()
    d_inv_sqrt = np.power(rowsum + 1e-5, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    return adj.dot(d_mat_inv_sqrt).transpose().dot(d_mat_inv_sqrt).tocoo()


#normalized_laplacian
def normalize_adj(adj):
    """Compute the normalized Laplacian."""
    adj = sp.coo_matrix(adj)
    rowsum = np.array(adj.sum(1)).flatten()
    d_inv_sqrt = np.power(rowsum + 1e-5, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    identity = sp.eye(adj.shape[0])  # Identity matrix of the same size as adj
    laplacian = identity - d_mat_inv_sqrt.dot(adj).dot(d_mat_inv_sqrt)  # Normalized Laplacian
    return laplacian.tocoo()


# normalization weight matrix:
def normalize_weight_matrix(W):
    """Symmetrically normalize the weight matrix."""
    W = sp.coo_matrix(W)
    rowsum = np.array(W.sum(1)).flatten()  # Sum of each row
    d_inv_sqrt = np.power(rowsum + 1e-5, -0.5)  # Inverse square root of row sums
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0  # Set infinities to 0
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)  # Diagonal matrix of d_inv_sqrt
    normalized_W = W.dot(d_mat_inv_sqrt).transpose().dot(d_mat_inv_sqrt)  # Symmetric normalization
    return normalized_W.tocoo()  # Return as sparse matrix (COO format)
#in_channels = 210  # Number of input features (e.g., time points or perturbations)
#hidden_channels = 64  # Hidden layer size
#out_channels = 32  # Output embedding size (could be adjusted based on your task)

# Instantiate the encoder

# Convert ground truth matrix to edge list
def convert_matrix_to_edge_list(matrix):
    edges = []
    for source in matrix.index:
        for target in matrix.columns:
            if matrix.at[source, target] == 1:
                edges.append((source, target))
    return edges


# Define the function to perform the GAE process for each threshold and multiple iterations
def run_gae_multiple_thresholds(data, ground_truth, algorithms, thresholds, num_iterations=30):
    num_genes = data.shape[0]
    #node_features = np.identity(num_genes)
    all_results = {}
    iteration_sums = {}

    for threshold in thresholds:
        print(f"Running for threshold: {threshold}")
        
        # Create a dictionary to store the results for each algorithm and threshold
        all_results[f'threshold_{threshold}'] = {}
        iteration_sums[f'threshold_{threshold}'] = {}

        # Initialize the iteration sum matrices for each threshold
        for iter_num in range(num_iterations):
            iteration_sums[f'threshold_{threshold}'][f'iter{iter_num + 1}_GCN_sum'] = np.zeros((num_genes, num_genes))

        for alg in algorithms:
            print(f"Running GAE for algorithm: {alg} with threshold: {threshold}")
            alg_columns = [col for col in data.columns if col.startswith(alg + '.')]
            alg_matrix = data[alg_columns].values

            # For each algorithm, create a list to store multiple runs
            all_results[f'threshold_{threshold}'][alg] = []

            for iteration in range(num_iterations):
                print(f"Iteration {iteration + 1}/{num_iterations} for algorithm: {alg}")
                
                # Apply GAE to the adjacency matrix of the algorithm
                adj_reconstructed, losses, accuracies = apply_gae_to_adjacency(alg_matrix, node_features)
                
                # Binarize the reconstructed adjacency matrix using the current threshold
                adj_binary = (adj_reconstructed > threshold).astype(int)

                # Store the binary adjacency matrix for this run
                all_results[f'threshold_{threshold}'][alg].append(adj_binary)

                # Add the binary matrix to the sum for this iteration and threshold
                iteration_sums[f'threshold_{threshold}'][f'iter{iteration + 1}_GCN_sum'] += adj_binary

    return all_results, iteration_sums

# Function to apply a second binarization threshold to the summed adjacency matrices
def apply_iteration_threshold(iteration_sums, thresholds, iter_thresholds):
    binary_sums = {}
    
    for threshold in thresholds:
        binary_sums[f'threshold_{threshold}'] = {}
        for iter_num in iteration_sums[f'threshold_{threshold}'].keys():
            binary_sums[f'threshold_{threshold}'][iter_num] = {}

            for iter_threshold in iter_thresholds:
                print(f"Applying iter_threshold {iter_threshold} to {iter_num} for threshold {threshold}")
                
                # Apply binarization on the summed adjacency matrix
                summed_matrix = iteration_sums[f'threshold_{threshold}'][iter_num]
                binary_matrix = (summed_matrix >= iter_threshold).astype(int)

                # Store the result
                binary_sums[f'threshold_{threshold}'][iter_num][f'iter_threshold_{iter_threshold}'] = binary_matrix
    
    return binary_sums

# Function to sum across iterations and apply a common threshold
def sum_across_iterations_and_binarize(binary_summed_matrices, thresholds, iter_thresholds, common_threshold):
    common_sums = {}

    for threshold in thresholds:
        common_sums[f'threshold_{threshold}'] = {}
        for iter_threshold in iter_thresholds:
            print(f"Summing across iterations for threshold {threshold} and iter_threshold {iter_threshold}")

            # Initialize the matrix for summing across iterations
            iteration_sum_matrix = np.zeros_like(binary_summed_matrices[f'threshold_{threshold}']['iter1_GCN_sum'][f'iter_threshold_{iter_threshold}'])

            # Sum the matrices across all iterations
            for iter_num in binary_summed_matrices[f'threshold_{threshold}'].keys():
                iteration_sum_matrix += binary_summed_matrices[f'threshold_{threshold}'][iter_num][f'iter_threshold_{iter_threshold}']

            # Apply the common threshold: if sum >= common_threshold, set 1; else set 0
            final_binarized_matrix = (iteration_sum_matrix >= common_threshold).astype(int)

            # Store the result
            common_sums[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}'] = final_binarized_matrix

    return common_sums


# Function to extract gene interactions (edges) from a binarized matrix
def extract_gene_interactions(final_matrix, gene_names):
    edges = []
    num_genes = final_matrix.shape[0]

    # Loop through the upper triangle of the matrix to avoid duplicates (since it's undirected)
    for i in range(num_genes):
        for j in range(i + 1, num_genes):
            if final_matrix[i, j] == 1:  # Check if there's an interaction
                edges.append(f"{gene_names[i]}-{gene_names[j]}")
    
    return edges


# Function to convert edges to consistent tuple format (e.g., ('G1', 'G10'))
def convert_edges_to_tuples(edge_list):
    edge_tuples = []
    for edge in edge_list:
        gene1, gene2 = edge.split('-')
        # Sort genes to maintain consistency in the edge representation (G1, G10) or (G10, G1)
        edge_tuples.append(tuple(sorted([gene1, gene2])))
    return set(edge_tuples)

# Function to compare predicted edges with known ground truth and generate the metrics
def compare_with_ground_truth(predicted_edges, known_edges_set):
    predicted_set = convert_edges_to_tuples(predicted_edges)
    
    # Calculate true positives and false positives
    true_positives = predicted_set.intersection(known_edges_set)
    false_positives = predicted_set.difference(known_edges_set)
    
    # Metrics
    num_true_positives = len(true_positives)
    num_false_positives = len(false_positives)
    
    return true_positives, false_positives, num_true_positives, num_false_positives

# Load the data
data = pd.read_csv("/Users/nirwantandukar/input_data/Database.DREAM4/NORMALIZED_EXP/normalized_weight_matrix_all_GRN_010_01_0.2filter.csv", index_col=0)
#data = pd.read_csv("/Users/nirwantandukar/input_data/weights_normalized/normalize_weight_matrix_all_GRN_100_01.csv", index_col=0)
print(f"Loaded data with shape: {data.shape}")

# Get the list of algorithms
columns = data.columns
algorithms = sorted(set(col.split('.')[0] for col in columns))
print(f"Algorithms found: {algorithms}")

num_genes = data.shape[0]
gene_names = data.index.tolist()
print(f"Number of genes: {num_genes}")

# Node features (identity matrix)
node_features = np.identity(num_genes)
#print(f"Node features shape: {node_features.shape}")


# Step 1: Load known (ground truth) interactions
file_path = "/Users/nirwantandukar/input_data/Database.DREAM4/GS/dream4_100_01_gs.csv"
ground_truth = pd.read_csv(file_path, index_col=0)
print(f"Ground truth loaded with shape: {ground_truth.shape}")

# Get the ground truth edges
known_edges_set = set(convert_matrix_to_edge_list(ground_truth))
print(f"Number of known edges: {len(known_edges_set)}")
# Define thresholds and number of iterations
thresholds = [0.6, 0.8]  # Different thresholds to run
#thresholds = [0.6]  # Different thresholds to run
num_iterations = 10  # Number of runs for each threshold

# Run the GAE process and store results for each threshold and algorithm
aggregated_results, iteration_sums = run_gae_multiple_thresholds(data, ground_truth, algorithms, thresholds, num_iterations)

# Define the additional binarization thresholds for the summed matrices
iter_thresholds = [2,4,6, 8,10,12,14]  # Example thresholds for binarizing summed matrices
#iter_thresholds = [4]  # Example thresholds for binarizing summed matrices

# Apply the iteration-level thresholding
binary_summed_matrices = apply_iteration_threshold(iteration_sums, thresholds, iter_thresholds)

# Example: Access binarized summed matrices for a specific iteration and iter_threshold
for threshold in thresholds:
    for iter_num in iteration_sums[f'threshold_{threshold}'].keys():
        for iter_threshold in iter_thresholds:
            binarized_matrix = binary_summed_matrices[f'threshold_{threshold}'][iter_num][f'iter_threshold_{iter_threshold}']
            print(f"Binarized adjacency matrix for {iter_num}, threshold {threshold}, iter_threshold {iter_threshold}:")
            print(binarized_matrix)

            # You can store or analyze these matrices further
            #output_file = f'binarized_{iter_num}_{threshold}_iter_thresh_{iter_threshold}.csv'
            #pd.DataFrame(binarized_matrix).to_csv(output_file)


# Define the common threshold
common_threshold = 8  # This is the threshold where if sum >= 30 across iterations, we set the value to 1. Consistently found in many iterations

# Sum across iterations and apply the common threshold
final_common_binarized_matrices = sum_across_iterations_and_binarize(binary_summed_matrices, thresholds, iter_thresholds, common_threshold)

# Example: Access and print the final binarized matrix for a specific threshold and iteration threshold
for threshold in thresholds:
    for iter_threshold in iter_thresholds:
        final_matrix = final_common_binarized_matrices[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}']
        print(f"Final binarized matrix for threshold {threshold}, iter_threshold {iter_threshold}, common_threshold {common_threshold}:")
        print(final_matrix)

        # You can save these matrices to CSV or further analyze them
        output_file = f'final_binarized_threshold_{threshold}_iter_thresh_{iter_threshold}_common_thresh_{common_threshold}.csv'
        #pd.DataFrame(final_matrix).to_csv(output_file)

# Assuming the gene names are the row/column names from your original data
gene_names = list(data.index)  # Replace 'data' with your actual DataFrame that contains the gene names

# Example: Use the final binarized matrix from the previous step (e.g., for threshold 0.8, iter_threshold 6, common_threshold 2)
for threshold in thresholds:
    for iter_threshold in iter_thresholds:
        final_matrix = final_common_binarized_matrices[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}']
        
        # Extract gene interactions (edges)
        edges = extract_gene_interactions(final_matrix, gene_names)
        
        # Print the edges
        print(f"Gene interactions for threshold {threshold}, iter_threshold {iter_threshold}, common_threshold {common_threshold}:")
        for edge in edges:
            print(edge)

        # Optionally, save the edges to a file
        output_file = f'gene_interactions_threshold_{threshold}_iter_thresh_{iter_threshold}_common_thresh_{common_threshold}.txt'
        with open(output_file, 'w') as f:
            for edge in edges:
                f.write(edge + '\n')

# Example structure to store the results
results = []


# Assume thresholds, iter_thresholds, and common_threshold are defined (as in the previous steps)
for threshold in thresholds:
    for iter_threshold in iter_thresholds:
        # For each final binarized matrix (e.g., threshold 0.8, iter_threshold 6, common_threshold 2)
        final_matrix = final_common_binarized_matrices[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}']
        
        # Extract gene interactions (predicted edges) from the final matrix
        predicted_edges = extract_gene_interactions(final_matrix, gene_names)
        
        # Compare predicted edges with the known ground truth edges
        true_positives, false_positives, num_true_positives, num_false_positives = compare_with_ground_truth(predicted_edges, known_edges_set)
        
        # Create a row for the results table
        results.append({
            'Iterations': num_iterations,
            'Common_Iterations': common_threshold,
            'Iter_Threshold': iter_threshold,
            'Individual_Threshold': threshold,
            'True_Positives': ','.join([f"{tp[0]}-{tp[1]}" for tp in true_positives]),  # Format as G1-G10
            'False_Positives': ','.join([f"{fp[0]}-{fp[1]}" for fp in false_positives]),  # Format as G1-G10
            'No_True_Positives': num_true_positives,
            'No_False_Positives': num_false_positives
        })

# Convert the results to a DataFrame for better visualization
results_df = pd.DataFrame(results)

# Display the results
print(results_df)

# Optionally, save the results to a CSV file
results_df.to_csv('true_false_positives_summary_GAT.csv', index=False)

# Open a file to save the formatted AUROC and AUPRC results
with open('auroc_auprc_results.txt', 'w') as f:
    # Example: Compute AUROC and AUPRC for each final binarized matrix
    for threshold in thresholds:
        for iter_threshold in iter_thresholds:
            final_matrix = final_common_binarized_matrices[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}']
            
            # Compute AUROC and AUPRC
            auroc, auprc = compute_roc_and_prc(final_matrix, known_edges_set, gene_names)
            
            # Format the result strings
            auroc_line = f"AUROC for threshold {threshold}, iter_threshold {iter_threshold}: {auroc}\n"
            auprc_line = f"AUPRC for threshold {threshold}, iter_threshold {iter_threshold}: {auprc}\n"
            
            # Write the formatted results to the file
            f.write(auroc_line)
            f.write(auprc_line)

# Optionally, print a message when the file is saved
print("AUROC and AUPRC results have been saved to 'auroc_auprc_results_GAT.txt'.")


Loaded data with shape: (10, 160)
Algorithms found: ['GRN_ARACNE', 'GRN_BC3NET', 'GRN_C3NET', 'GRN_CLR', 'GRN_GENIE3_ET', 'GRN_GENIE3_RF', 'GRN_GRNBOOST2', 'GRN_INFERELATOR', 'GRN_KBOOST', 'GRN_LEAP', 'GRN_MEOMI', 'GRN_MRNET', 'GRN_MRNETB', 'GRN_NONLINEARODES', 'GRN_PCIT', 'GRN_TIGRESS']
Number of genes: 10
Ground truth loaded with shape: (100, 100)
Number of known edges: 175
Running for threshold: 0.6
Running GAE for algorithm: GRN_ARACNE with threshold: 0.6
Iteration 1/10 for algorithm: GRN_ARACNE


UnboundLocalError: cannot access local variable 'x' where it is not associated with a value

In [13]:
results_df

,Iterations,Common_Iterations,Iter_Threshold,Individual_Threshold,True_Positives,False_Positives,No_True_Positives,No_False_Positives
0,10,8,2,0.6,"G5-G8,G1-G4,G5-G9,G5-G6","G10-G3,G2-G7,G4-G6,G10-G4,G3-G4,G1-G5,G2-G6,G4...",4,26
1,10,8,4,0.6,G5-G9,"G10-G9,G2-G8,G2-G7,G2-G4,G6-G8,G2-G3,G3-G4,G1-...",1,11
2,10,8,6,0.6,,"G6-G8,G3-G4,G1-G5,G3-G7,G4-G7",0,5
3,10,8,8,0.6,,"G6-G8,G3-G4,G1-G5,G3-G7,G4-G7",0,5
4,10,8,10,0.6,,G1-G5,0,1
5,10,8,12,0.6,,,0,0
6,10,8,14,0.6,,,0,0
7,10,8,2,0.8,G5-G9,"G10-G9,G2-G8,G6-G8,G2-G3,G3-G4,G1-G5,G2-G6,G4-...",1,10
8,10,8,4,0.8,,"G1-G5,G3-G7,G3-G4,G6-G8",0,4
9,10,8,6,0.8,,"G1-G5,G6-G8",0,2


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax

class GATLayer(MessagePassing):
    def __init__(self, in_features, out_features, heads=4, dropout=0.1):
        super(GATLayer, self).__init__(aggr='add')  # "Add" aggregation for message passing
        self.heads = heads
        self.in_features = in_features
        self.out_features = out_features
        self.dropout = dropout

        # Learnable weight matrices for feature transformation
        self.W = torch.nn.Parameter(torch.Tensor(in_features, heads * out_features))  # Weight matrix for features
        self.attention = torch.nn.Parameter(torch.Tensor(1, heads, out_features))  # Learnable attention coefficients

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.xavier_uniform_(self.W)
        torch.nn.init.xavier_uniform_(self.attention)

    def forward(self, x, edge_index):
        # Step 1: Apply the linear transformation
        h = torch.matmul(x, self.W)  # Shape: (num_nodes, heads * out_features)
        h = h.view(-1, self.heads, self.out_features)  # Reshape for multiple heads

        # Step 2: Start message passing to calculate attention coefficients
        return self.propagate(edge_index, x=h)

    def message(self, x_i, x_j, edge_index, size):
        # x_i and x_j are the source and target node features, respectively
        # Step 3: Compute attention scores
        alpha = (x_i * self.attention).sum(dim=-1)  # Dot product between x_i and attention coefficients
        alpha = F.leaky_relu(alpha, negative_slope=0.2)  # LeakyReLU activation on attention scores

        # Step 4: Normalize attention scores using softmax
        alpha = softmax(alpha, edge_index[0], num_nodes=size[0])  # Apply softmax for normalization
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)  # Dropout for regularization

        # Step 5: Message passing: Multiply attention scores with neighbor features
        return x_j * alpha.unsqueeze(-1)  # Shape: (num_edges, heads, out_features)

    def update(self, aggr_out):
        # Step 6: Aggregate the updated node features after message passing
        return aggr_out.mean(dim=1)  # Average over heads

import torch
import torch.nn.functional as F
from torch_geometric.nn import BatchNorm

class GATEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, heads=4, dropout_rate=0.2):
        super(GATEncoder, self).__init__()
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate

        # First layer: input to hidden size with multi-head attention
        self.layers = torch.nn.ModuleList()
        self.layers.append(GATLayer(in_channels, hidden_channels, heads=heads, dropout=dropout_rate))

        # Add intermediate layers with attention
        for _ in range(num_layers - 2):
            self.layers.append(GATLayer(heads * hidden_channels, hidden_channels, heads=heads, dropout=dropout_rate))

        # Final layer: hidden size to output size
        self.layers.append(GATLayer(heads * hidden_channels, out_channels, heads=1, dropout=dropout_rate))  # Single head for final layer

        # Batch normalization for each layer
        self.bns = torch.nn.ModuleList([BatchNorm(hidden_channels * heads) for _ in range(num_layers - 1)])
    
        self.dropout = torch.nn.Dropout(p=dropout_rate)
        self.activation = torch.nn.LeakyReLU(negative_slope=0.05)

    def forward(self, x, edge_index):
        dead_neurons = []  # To track dead neurons across layers

        # Apply GAT layers with attention
        for i, layer in enumerate(self.layers):
            if i == 0:
                x = layer(x, edge_index)
            else:
                x_residual = x
                x = layer(x, edge_index)
                if i < len(self.bns):
                    x = self.bns[i-1](x)  # Batch normalization for intermediate layers
                x = self.activation(x)
                x = self.dropout(x)
                x = x + x_residual  # Add residual connection

        return x
def apply_gae_to_adjacency(adj_matrix, node_features, embedding_dim=8, epochs=300, dropout_rate=0.2, weight_decay=1e-4):
    # Ensure node features match the number of nodes in adj_matrix
    num_nodes = adj_matrix.shape[0]
    if node_features.shape[0] != num_nodes:
        raise ValueError(f"Node features should have {num_nodes} rows to match adjacency matrix!")

    # Convert adjacency matrix to sparse format
    adj_sparse = sp.csr_matrix(adj_matrix)
    adj_normalized = normalize_adj(adj_sparse + sp.eye(adj_sparse.shape[0]))
    edge_index, edge_weight = from_scipy_sparse_matrix(adj_normalized)

    # Check that edge indices are valid for node feature matrix
    if edge_index.max() >= num_nodes:
        raise IndexError(f"Found indices in 'edge_index' larger than {num_nodes - 1}. Check the node feature matrix size.")
    
    print(f"edge_index is valid with max index: {edge_index.max()} and node count: {num_nodes}")
    
    # Create data object
    x = torch.tensor(node_features, dtype=torch.float)
    data = Data(x=x, edge_index=edge_index)

    # Define the GATEncoder as the encoder for the GAE model
    encoder = GATEncoder(in_channels=x.shape[1], hidden_channels=embedding_dim, out_channels=embedding_dim, dropout_rate=dropout_rate)

    # Instantiate the GAE model by passing the encoder
    model = GAE(encoder)

    # Training settings
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    data = data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=weight_decay)

    # Metrics to track
    losses = []
    accuracies = []

    # Training loop
    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()

        # Forward pass
        z = model.encode(data.x, data.edge_index)

        # Reconstruction loss
        loss = model.recon_loss(z, data.edge_index)

        # Accuracy: Compare reconstruction with ground truth adjacency matrix
        adj_pred = (model.decoder.forward_all(z) > 0.5).float()  # Example threshold for prediction
        acc = (adj_pred == torch.tensor(adj_matrix).float()).sum() / adj_pred.numel()  # Accuracy for all elements
        
        # Backward pass
        loss.backward()
        optimizer.step()

        # Store metrics
        losses.append(loss.item())
        accuracies.append(acc.item())

        # Print metrics every 50 epochs
        if epoch % 50 == 0:
            print(f'Epoch {epoch} | Loss: {loss.item():.4f} | Accuracy: {acc.item() * 100:.2f}%')

    # After training, reconstruct adjacency matrix
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index)
        adj_reconstructed = model.decoder.forward_all(z).cpu().numpy()
    
    return adj_reconstructed, losses, accuracies




# Load the data
data = pd.read_csv("/Users/nirwantandukar/input_data/Database.DREAM4/NORMALIZED_EXP/normalized_weight_matrix_all_GRN_100_01_0.2filter.csv", index_col=0)
#data = pd.read_csv("/Users/nirwantandukar/input_data/weights_normalized/normalize_weight_matrix_all_GRN_100_01.csv", index_col=0)
print(f"Loaded data with shape: {data.shape}")

# Get the list of algorithms
columns = data.columns
algorithms = sorted(set(col.split('.')[0] for col in columns))
print(f"Algorithms found: {algorithms}")

num_genes = data.shape[0]
gene_names = data.index.tolist()
print(f"Number of genes: {num_genes}")

# Node features (identity matrix)
node_features = np.identity(num_genes)
print(f"Node features shape: {node_features.shape}")


# Step 1: Load known (ground truth) interactions
file_path = "/Users/nirwantandukar/input_data/Database.DREAM4/GS/dream4_100_01_gs.csv"
ground_truth = pd.read_csv(file_path, index_col=0)
print(f"Ground truth loaded with shape: {ground_truth.shape}")

# Get the ground truth edges
known_edges_set = set(convert_matrix_to_edge_list(ground_truth))
print(f"Number of known edges: {len(known_edges_set)}")
# Define thresholds and number of iterations
#thresholds = [0, 0.2,0.4,0.6, 0.8]  # Different thresholds to run
thresholds = [0.6]  # Different thresholds to run
num_iterations = 30  # Number of runs for each threshold

# Run the GAE process and store results for each threshold and algorithm
aggregated_results, iteration_sums = run_gae_multiple_thresholds(data, ground_truth, algorithms, thresholds, num_iterations)

# Define the additional binarization thresholds for the summed matrices
#iter_thresholds = [2,4,6, 8,10,12,14,16]  # Example thresholds for binarizing summed matrices
iter_thresholds = [4]  # Example thresholds for binarizing summed matrices

# Apply the iteration-level thresholding
binary_summed_matrices = apply_iteration_threshold(iteration_sums, thresholds, iter_thresholds)

# Example: Access binarized summed matrices for a specific iteration and iter_threshold
for threshold in thresholds:
    for iter_num in iteration_sums[f'threshold_{threshold}'].keys():
        for iter_threshold in iter_thresholds:
            binarized_matrix = binary_summed_matrices[f'threshold_{threshold}'][iter_num][f'iter_threshold_{iter_threshold}']
            print(f"Binarized adjacency matrix for {iter_num}, threshold {threshold}, iter_threshold {iter_threshold}:")
            print(binarized_matrix)

            # You can store or analyze these matrices further
            #output_file = f'binarized_{iter_num}_{threshold}_iter_thresh_{iter_threshold}.csv'
            #pd.DataFrame(binarized_matrix).to_csv(output_file)


# Define the common threshold
common_threshold = 30  # This is the threshold where if sum >= 30 across iterations, we set the value to 1. Consistently found in many iterations

# Sum across iterations and apply the common threshold
final_common_binarized_matrices = sum_across_iterations_and_binarize(binary_summed_matrices, thresholds, iter_thresholds, common_threshold)

# Example: Access and print the final binarized matrix for a specific threshold and iteration threshold
for threshold in thresholds:
    for iter_threshold in iter_thresholds:
        final_matrix = final_common_binarized_matrices[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}']
        print(f"Final binarized matrix for threshold {threshold}, iter_threshold {iter_threshold}, common_threshold {common_threshold}:")
        print(final_matrix)

        # You can save these matrices to CSV or further analyze them
        output_file = f'final_binarized_threshold_{threshold}_iter_thresh_{iter_threshold}_common_thresh_{common_threshold}.csv'
        #pd.DataFrame(final_matrix).to_csv(output_file)

# Assuming the gene names are the row/column names from your original data
gene_names = list(data.index)  # Replace 'data' with your actual DataFrame that contains the gene names

# Example: Use the final binarized matrix from the previous step (e.g., for threshold 0.8, iter_threshold 6, common_threshold 2)
for threshold in thresholds:
    for iter_threshold in iter_thresholds:
        final_matrix = final_common_binarized_matrices[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}']
        
        # Extract gene interactions (edges)
        edges = extract_gene_interactions(final_matrix, gene_names)
        
        # Print the edges
        print(f"Gene interactions for threshold {threshold}, iter_threshold {iter_threshold}, common_threshold {common_threshold}:")
        for edge in edges:
            print(edge)

        # Optionally, save the edges to a file
        output_file = f'gene_interactions_threshold_{threshold}_iter_thresh_{iter_threshold}_common_thresh_{common_threshold}.txt'
        with open(output_file, 'w') as f:
            for edge in edges:
                f.write(edge + '\n')

# Example structure to store the results
results = []


# Assume thresholds, iter_thresholds, and common_threshold are defined (as in the previous steps)
for threshold in thresholds:
    for iter_threshold in iter_thresholds:
        # For each final binarized matrix (e.g., threshold 0.8, iter_threshold 6, common_threshold 2)
        final_matrix = final_common_binarized_matrices[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}']
        
        # Extract gene interactions (predicted edges) from the final matrix
        predicted_edges = extract_gene_interactions(final_matrix, gene_names)
        
        # Compare predicted edges with the known ground truth edges
        true_positives, false_positives, num_true_positives, num_false_positives = compare_with_ground_truth(predicted_edges, known_edges_set)
        
        # Create a row for the results table
        results.append({
            'Iterations': num_iterations,
            'Common_Iterations': common_threshold,
            'Iter_Threshold': iter_threshold,
            'Individual_Threshold': threshold,
            'True_Positives': ','.join([f"{tp[0]}-{tp[1]}" for tp in true_positives]),  # Format as G1-G10
            'False_Positives': ','.join([f"{fp[0]}-{fp[1]}" for fp in false_positives]),  # Format as G1-G10
            'No_True_Positives': num_true_positives,
            'No_False_Positives': num_false_positives
        })

# Convert the results to a DataFrame for better visualization
results_df = pd.DataFrame(results)

# Display the results
print(results_df)

# Optionally, save the results to a CSV file
results_df.to_csv('true_false_positives_summary_weight_6.csv', index=False)



In [57]:
node_features = np.identity(num_genes)
print(f"Node features shape: {node_features.shape}")


Node features shape: (100, 100)


In [61]:
assert edge_index.max() < node_features.shape[0], "Invalid edge_index; node indices exceed number of nodes!"


NameError: name 'edge_index' is not defined

In [65]:
print(f"x.shape: {x.shape}")  # It should be (100, num_features)


NameError: name 'x' is not defined